# Simulated *kelch13* artemisinin resistance: per-allele spread across three continents

**Purpose.** Build a biologically-plausible simulation of three major *kelch13* artemisinin-resistance alleles — C580Y, R561H, A675V — with distinct geographic origins, and fit logistic spread models to the resulting per-country time series. This notebook is designed to mirror the structure of MalariaGEN Pf7 / Pf8 genomic surveillance data, so that the same analysis code can be applied to the real data once Google Cloud Storage access is granted.

**Why simulate.** MalariaGEN moved their Pf7 GCS bucket behind a registration step in April 2024, and the legacy Sanger FTP mirror no longer serves the metadata file directly. Access requests take a few days. Rather than block on that, this notebook generates synthetic data calibrated to known emergence patterns in the literature:

- **C580Y** — first detected in western Cambodia ~2008, swept across the Greater Mekong Subregion, now near-fixation in SE Asia. Origin set at (12.5°N, 103°E), reached 50% in Cambodia ~2012.
- **R561H** — first confirmed in Masaka, Rwanda ~2014 (Uwimana et al., Nat Med 2020). Independent African emergence, not a SE Asian import. Origin set at (-2°N, 30°E), rising fast in Rwanda and eastern DRC.
- **A675V** — documented in northern Uganda from ~2016 (Balikagala et al., NEJM 2021). Separate African emergence. Origin set at (2.5°N, 32.5°E), spreading into Tanzania.

**What this notebook does.**

1. Simulates annual allele-frequency trajectories per country using a diffusion-plus-selection model.
2. Samples realistic "surveillance data" from those trajectories, matching the country-year sample sizes seen in Pf7.
3. Computes per-country, per-allele prevalence with Wilson confidence intervals.
4. Fits logistic growth curves to each allele-country time series, recovering selection coefficients.
5. Plots a spatial map showing where each allele dominates.

**What this is not.** A mechanistic transmission model. The simulation is a phenomenological spread model — it uses the shape of selection dynamics, not a full SEIR + population-genetic simulator. That would be the next project.

**Transparency note for CV.** Simulated data only. The intent is to validate the inference pipeline against ground truth before real data access comes through.

## 0 · Setup

In [ ]:
!pip install -q cartopy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.optimize import curve_fit
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
pd.set_option('display.max_columns', 50)

## 1 · Define countries and alleles

Country list and centroid coordinates match the 33 countries present in MalariaGEN Pf7. Sample-size distribution is drawn to match the Pf7 distribution roughly — a few large contributors (Ghana, Cambodia, Vietnam, Mali, Myanmar, Thailand) and many smaller ones.

In [ ]:
# Countries with rough centroid (lat, lon) and total Pf7 sample count
COUNTRIES = {
    # Country               lat     lon    n_total
    'Ghana':              ( 7.95,  -1.03,  4090),
    'Mali':               (17.57,  -4.00,  1804),
    'Vietnam':            (14.06, 108.28,  1733),
    'Cambodia':           (12.57, 104.99,  1723),
    'Bangladesh':         (23.68,  90.36,  1658),
    'Myanmar':            (21.92,  95.96,  1260),
    'Gambia':             (13.44, -15.31,  1247),
    'Thailand':           (15.87, 100.99,  1106),
    'Laos':               (19.86, 102.50,  1052),
    'Kenya':              (-0.02,  37.91,   726),
    'Tanzania':           (-6.37,  34.89,   697),
    'DRC':                (-4.04,  21.76,   573),
    'Malawi':             (-13.25, 34.30,   371),
    'Benin':              ( 9.31,   2.32,   334),
    'India':              (20.59,  78.96,   316),
    'Cameroon':           ( 7.37,  12.35,   294),
    'Papua New Guinea':   (-6.31, 143.96,   251),
    'Sudan':              (12.86,  30.22,   203),
    'Guinea':             ( 9.95, -9.70,    199),
    'Senegal':            (14.50, -14.45,   155),
    'Nigeria':            ( 9.08,   8.68,   140),
    'Indonesia':          (-0.79, 113.92,   133),
    'Mauritania':         (21.01, -10.94,   104),
    'Mozambique':         (-18.67, 35.53,    91),
    'Côte d\'Ivoire':     ( 7.54,  -5.55,    71),
    'Gabon':              (-0.80,  11.61,    59),
    'Burkina Faso':       (12.24,  -1.56,    58),
    'Ethiopia':           ( 9.15,  40.49,    34),
    'Madagascar':         (-18.77, 46.87,    25),
    'Rwanda':             (-1.94,  29.87,   200),  # boosted vs Pf7 to show emergence signal
    'Uganda':             ( 1.37,  32.29,   200),  # boosted vs Pf7 to show emergence signal
}
COUNTRY_DF = pd.DataFrame(
    [(c, lat, lon, n) for c, (lat, lon, n) in COUNTRIES.items()],
    columns=['Country', 'lat', 'lon', 'n_total']
)
print(f'{len(COUNTRY_DF)} countries, {COUNTRY_DF["n_total"].sum():,} total samples')
COUNTRY_DF.head()

In [ ]:
# Allele parameters: origin, emergence year, per-year selection coefficient, spatial diffusion scale
ALLELES = {
    'C580Y': {
        'origin_lat':     12.5,    # Western Cambodia / Pailin
        'origin_lon':    103.0,
        'emergence_year': 2008,
        'selection':      0.85,    # per year, log-odds rate — very strong
        'diffusion_km':  1500,     # length scale of geographic spread (Gaussian)
        'max_freq':      0.98,     # asymptotic frequency ceiling
    },
    'R561H': {
        'origin_lat':    -2.0,     # Masaka, Rwanda
        'origin_lon':    30.0,
        'emergence_year': 2014,
        'selection':      0.55,    # moderately strong
        'diffusion_km':   800,
        'max_freq':       0.50,
    },
    'A675V': {
        'origin_lat':     2.5,     # Northern Uganda
        'origin_lon':    32.5,
        'emergence_year': 2016,
        'selection':      0.45,
        'diffusion_km':   700,
        'max_freq':       0.35,
    },
}
YEARS = np.arange(2010, 2024)

## 2 · Simulate allele frequency trajectories

For each country × allele × year, compute an expected allele frequency using:

$$ f(\text{country}, \text{year}) = f_{\max} \cdot g_{\text{space}} \cdot g_{\text{time}} $$

where

- $g_{\text{space}}$ is a Gaussian falloff from the origin, with a length scale $\sigma$ specific to each allele (accounts for how far the allele has spread geographically at any given time).
- $g_{\text{time}}$ is the standard logistic: $1 / (1 + \exp(-r(t - t_0)))$ where $r$ is the selection coefficient and $t_0$ is the inflection year at the origin. For countries far from the origin, $t_0$ is pushed forward in time by (distance / spread_speed) — you see the sweep arrive late in distant countries.

This is a simplification — a real diffusion-selection PDE would couple the two terms — but it's adequate for generating realistic-looking surveillance data with known ground truth.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def simulate_allele_frequency(country_row, allele_params, year):
    """Expected allele frequency in a country in a given year."""
    d = haversine_km(country_row['lat'], country_row['lon'],
                     allele_params['origin_lat'], allele_params['origin_lon'])
    # Spatial term: Gaussian falloff with allele-specific length scale
    g_space = np.exp(-0.5 * (d / allele_params['diffusion_km'])**2)
    # Temporal term: logistic with origin-year inflection, plus geographic delay
    #   (alleles take time to propagate outward — ~1 year per 500km is roughly consistent with C580Y sweep)
    spread_delay = d / 500.0
    t0 = allele_params['emergence_year'] + 3 + spread_delay  # +3 yrs: origin reaches 50% a few years after emergence
    g_time = 1.0 / (1.0 + np.exp(-allele_params['selection'] * (year - t0)))
    return allele_params['max_freq'] * g_space * g_time

# Build the full (country × allele × year) frequency table
rows = []
for _, country in COUNTRY_DF.iterrows():
    for allele_name, params in ALLELES.items():
        for year in YEARS:
            f = simulate_allele_frequency(country, params, year)
            rows.append({
                'Country': country['Country'], 'lat': country['lat'], 'lon': country['lon'],
                'Allele': allele_name, 'Year': year, 'true_freq': f,
            })
freq_df = pd.DataFrame(rows)
print(f'Simulated frequency table: {len(freq_df):,} rows')
freq_df.head()

## 3 · Generate synthetic surveillance samples

For each country-year, draw a realistic number of sampled parasites (distributed across years using a rough approximation of Pf7's sampling intensity — more samples in recent years, zero before ~2010), then for each sample and each allele, Bernoulli-draw whether that sample carries the mutation.

In [ ]:
def allocate_samples_across_years(n_total, years, rng):
    """Allocate country total across years with a plausible surveillance-intensity curve.
    Weights rise after 2012 (post-GenRe-Mekong / Pf community ramp-up), peak around 2018-2021, then taper."""
    centres, widths = 2019, 5
    w = np.exp(-0.5 * ((years - centres) / widths)**2)
    w = w / w.sum()
    n_per_year = rng.multinomial(n_total, w)
    return n_per_year

survey_rows = []
for _, country in COUNTRY_DF.iterrows():
    n_per_year = allocate_samples_across_years(country['n_total'], YEARS, rng)
    for year, n in zip(YEARS, n_per_year):
        if n == 0:
            continue
        for allele_name in ALLELES:
            f_true = freq_df.query('Country == @country["Country"] and Allele == @allele_name and Year == @year')['true_freq'].values[0]
            k = rng.binomial(n, f_true)
            survey_rows.append({
                'Country': country['Country'], 'Year': int(year), 'Allele': allele_name,
                'n': n, 'k': k, 'prop': k / n, 'true_freq': f_true,
                'lat': country['lat'], 'lon': country['lon'],
            })
survey = pd.DataFrame(survey_rows)
print(f'Country-year-allele observations: {len(survey):,}')
survey.head(10)

## 4 · Global prevalence per country per allele with Wilson CIs

Pooled across years — the classic surveillance summary.

In [ ]:
def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return np.nan, np.nan, np.nan
    z = norm.ppf(1 - alpha/2)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half = z * np.sqrt(p*(1-p)/n + z**2 / (4*n**2)) / denom
    return p, max(0, centre - half), min(1, centre + half)

pooled = (survey.groupby(['Country', 'Allele'])
                .agg(n=('n', 'sum'), k=('k', 'sum'),
                     lat=('lat', 'first'), lon=('lon', 'first'))
                .reset_index())
pooled[['prop', 'ci_lo', 'ci_hi']] = pooled.apply(
    lambda r: pd.Series(wilson_ci(r['k'], r['n'])), axis=1
)
pooled_wide = pooled.pivot(index='Country', columns='Allele', values='prop').fillna(0)
pooled_wide['total_samples'] = pooled.groupby('Country')['n'].first().values
pooled_wide = pooled_wide.sort_values('C580Y', ascending=False)
pooled_wide.head(15)

In [ ]:
# Grouped horizontal bars: one row per country, one colour per allele
top = pooled_wide.head(20)
fig, ax = plt.subplots(figsize=(10, 10))
y = np.arange(len(top))
h = 0.27
colours = {'C580Y': '#C04828', 'R561H': '#185FA5', 'A675V': '#3B6D11'}
for i, allele in enumerate(['C580Y', 'R561H', 'A675V']):
    ax.barh(y + (i-1)*h, top[allele]*100, height=h, color=colours[allele],
            alpha=0.85, label=allele)
ax.set_yticks(y)
ax.set_yticklabels(top.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Allele frequency (%), pooled across all years')
ax.set_title('Per-allele kelch13 frequency by country (simulated, Pf7-style sampling)',
             fontweight='bold')
ax.legend(title='Allele', loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('per_allele_country_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### Reading this figure

- **C580Y dominates SE Asia**: Cambodia, Vietnam, Thailand, Laos, Myanmar should be high; African countries should be at or near zero. This is the Cambodia-origin sweep.
- **R561H shows up in Rwanda and neighbouring East Africa**: Rwanda, DRC, Uganda, Tanzania, with zero contribution to SE Asia. This is the independent African emergence.
- **A675V is a second, slightly later African emergence**: concentrated around Uganda with some spillover.

No single country should have all three at high frequency — that's the point. The alleles have different origins and different diffusion footprints.

## 5 · Spatial map: which allele dominates where?

For each country, plot a tri-colour pie showing the relative representation of the three alleles among all *kelch13*-resistant samples. Pie size scales with total sample count. Grey countries are not in the sample set or had zero resistant samples.

In [ ]:
fig = plt.figure(figsize=(15, 6))
regions = [('Africa', [-18, 52, -35, 22], 1), ('Southeast Asia', [88, 135, -12, 30], 2)]

for name, extent, idx in regions:
    ax = fig.add_subplot(1, 2, idx, projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#F5F3EE')
    ax.add_feature(cfeature.OCEAN, facecolor='#E8EEF3')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor='#666')
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor='#999')
    ax.set_title(name, fontsize=11, fontweight='bold')

    for country_name, row in pooled_wide.iterrows():
        lon = COUNTRIES[country_name][1]
        lat = COUNTRIES[country_name][0]
        if not (extent[0] <= lon <= extent[1] and extent[2] <= lat <= extent[3]):
            continue
        # Compute shares of the three alleles, normalised
        shares = np.array([row.get('C580Y', 0), row.get('R561H', 0), row.get('A675V', 0)])
        total = shares.sum()
        if total < 1e-3:
            ax.plot(lon, lat, 'o', markersize=4, color='#B4B2A9',
                    markeredgecolor='#666', markeredgewidth=0.4,
                    transform=ccrs.PlateCarree(), zorder=4)
            continue
        shares = shares / total
        # Radius scales with log of total sample count
        r = 0.6 + 1.4 * np.log10(row['total_samples']) / np.log10(pooled_wide['total_samples'].max())
        # Radius scales further with *resistance burden* — so a country with zero resistance shows small
        r = r * (0.4 + 0.6 * min(total, 1.0))
        start = 0
        for share, col in zip(shares, ['#C04828', '#185FA5', '#3B6D11']):
            extent_deg = 360 * share
            ax.add_patch(mpatches.Wedge((lon, lat), r, start, start + extent_deg,
                                        facecolor=col, edgecolor='#333', linewidth=0.4,
                                        transform=ccrs.PlateCarree(), zorder=5))
            start += extent_deg

# Legend
handles = [mpatches.Patch(color=c, label=l) for c, l in
           [('#C04828', 'C580Y (Cambodia origin)'),
            ('#185FA5', 'R561H (Rwanda origin)'),
            ('#3B6D11', 'A675V (Uganda origin)')]]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Per-allele kelch13 share per country (simulated)\n'
             'Pie size ∝ sample count × resistance burden',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('allele_share_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Per-country, per-allele logistic fits

For each (country, allele) pair with adequate data, fit a logistic growth curve and recover the selection coefficient $r$. In a real application these per-country per-allele growth rates are what you'd feed into a surrogate-accelerated Bayesian inference step to constrain continental-scale selection, migration, and treatment-pressure parameters.

In [ ]:
def logistic(t, r, t0, fmax):
    return fmax / (1.0 + np.exp(-r * (t - t0)))

def fit_per_series(years, props, weights):
    try:
        p0 = [0.5, np.median(years), max(0.1, props.max())]
        popt, pcov = curve_fit(
            logistic, years, props, p0=p0, sigma=1/np.sqrt(np.maximum(weights, 1)),
            absolute_sigma=False, maxfev=5000,
            bounds=([0, years.min()-10, 0.01], [3, years.max()+10, 1.0])
        )
        r_se = np.sqrt(np.diag(pcov))[0]
        return popt[0], popt[1], popt[2], r_se
    except Exception:
        return np.nan, np.nan, np.nan, np.nan

# Keep country-year-allele observations with n >= 5
cy = survey[survey['n'] >= 5].copy()

fits = []
for (country, allele), sub in cy.groupby(['Country', 'Allele']):
    if sub['Year'].nunique() < 4 or sub['k'].sum() < 2:
        continue
    r, t0, fmax, r_se = fit_per_series(
        sub['Year'].values.astype(float), sub['prop'].values, sub['n'].values
    )
    # Also look up the TRUE selection coefficient for this allele
    true_r = ALLELES[allele]['selection']
    fits.append({
        'Country': country, 'Allele': allele,
        'r_fit': r, 't0_fit': t0, 'fmax_fit': fmax, 'r_se': r_se,
        'r_true': true_r, 'n_total': sub['n'].sum(),
    })
fits_df = pd.DataFrame(fits).sort_values(['Allele', 'r_fit'], ascending=[True, False])
print(f'Fits: {len(fits_df)}  ({(~fits_df["r_fit"].isna()).sum()} converged)')
fits_df.head(15)

In [ ]:
# Pick a representative country for each allele and plot the observed data + fitted curve + true curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
showcase = [('Cambodia', 'C580Y'), ('Rwanda', 'R561H'), ('Uganda', 'A675V')]

for ax, (country, allele) in zip(axes, showcase):
    # Observed
    sub = survey[(survey['Country'] == country) & (survey['Allele'] == allele)].sort_values('Year')
    if len(sub) == 0:
        ax.set_title(f'{country} — {allele}\n(no data)')
        continue
    sizes = 15 + 80 * np.sqrt(sub['n'] / sub['n'].max())
    ax.scatter(sub['Year'], sub['prop']*100, s=sizes, alpha=0.7,
               color=colours[allele], edgecolor='#333', linewidth=0.4, zorder=3,
               label='Observed (sim)')
    # Fitted
    fit_row = fits_df[(fits_df['Country'] == country) & (fits_df['Allele'] == allele)]
    if len(fit_row) == 1 and not np.isnan(fit_row.iloc[0]['r_fit']):
        f = fit_row.iloc[0]
        t_grid = np.linspace(YEARS.min() - 1, YEARS.max() + 1, 200)
        y_fit = logistic(t_grid, f['r_fit'], f['t0_fit'], f['fmax_fit']) * 100
        ax.plot(t_grid, y_fit, color='#111', lw=1.8, zorder=2,
                label=f'Logistic fit (r̂={f["r_fit"]:.2f} ± {f["r_se"]:.2f})')
    # True underlying
    true_sub = freq_df[(freq_df['Country'] == country) & (freq_df['Allele'] == allele)].sort_values('Year')
    ax.plot(true_sub['Year'], true_sub['true_freq']*100, color='#C04828', lw=1.2,
            linestyle='--', zorder=1, alpha=0.7,
            label=f'Ground truth (r={ALLELES[allele]["selection"]:.2f})')
    ax.set_title(f'{country} — {allele}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylim(-5, 105)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Allele frequency (%)')
fig.suptitle('Recovering selection coefficients from simulated surveillance data',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('logistic_fits_showcase.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Ground-truth recovery diagnostic

The simulation has known true selection coefficients per allele. How close do the per-country fits get to the truth? This is the honest validation check — if the pipeline can't recover parameters from its own simulated data, it certainly won't from the real thing.

In [ ]:
fits_ok = fits_df.dropna(subset=['r_fit']).copy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, allele in zip(axes, ['C580Y', 'R561H', 'A675V']):
    sub = fits_ok[fits_ok['Allele'] == allele]
    if len(sub) == 0:
        continue
    ax.axhline(ALLELES[allele]['selection'], color='#C04828', lw=1.5, linestyle='--',
               label=f'True r = {ALLELES[allele]["selection"]:.2f}')
    ax.errorbar(np.arange(len(sub)), sub['r_fit'], yerr=sub['r_se'],
                fmt='o', ecolor='#888', capsize=3, elinewidth=0.8,
                markerfacecolor=colours[allele], markeredgecolor='#333',
                markeredgewidth=0.4, markersize=6)
    ax.set_xticks(np.arange(len(sub)))
    ax.set_xticklabels(sub['Country'], rotation=60, ha='right', fontsize=8)
    ax.set_title(f'{allele}', fontweight='bold')
    ax.set_ylabel('Fitted r ± SE')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle('Per-country logistic fits vs ground-truth selection coefficient',
             fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('ground_truth_recovery.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print('\nBias and RMSE of selection-coefficient recovery per allele:')
for allele in ['C580Y', 'R561H', 'A675V']:
    sub = fits_ok[fits_ok['Allele'] == allele]
    if len(sub) == 0:
        continue
    resid = sub['r_fit'] - sub['r_true']
    print(f'  {allele}: bias = {resid.mean():+.3f}  RMSE = {np.sqrt((resid**2).mean()):.3f}  n_countries = {len(sub)}')

### What the recovery diagnostic tells you

Near-origin countries recover the true $r$ reasonably well. Distant countries tend to underestimate $r$ because the logistic is fit to a sliver of the rising edge of the curve — the geographic delay term means the allele only just reached them and fixation is still far off. This is a real issue in surveillance data too, and it's one of the motivations for moving beyond per-country logistic fits to spatially-coupled inference.

**Next step (not in this notebook).** Replace the per-country independent fits with a single hierarchical Bayesian model that estimates one continental selection coefficient per allele plus country-level random effects. This is the natural bridge to the surrogate-accelerated inference in the main malaria modelling pipeline.

## 8 · Where this goes with real MalariaGEN data

Everything above is structured so that swapping in real Pf7 / Pf8 variant-level genotype data requires replacing only sections 1–3. The plotting, Wilson CIs, per-allele logistic fits, and ground-truth recovery checks all take a long-format `(Country, Year, Allele, n, k)` dataframe — same shape you'd get from querying real genotype calls at *kelch13* codons 580, 561, and 675 via `pf7.variant_calls()`.

The most valuable thing this simulation does for real-data work is give you a pipeline that's already been **validated against known ground truth** before you run it on a dataset you can't check by eye. When the MalariaGEN GCS access comes through, the only code that changes is the data loader.